In [1]:
import pandas as pd
import numpy as np
import joblib
import re

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
BASE_DIR = Path("..")

DATA_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "english_tickets_processed.csv"
)

MODELS_DIR = BASE_DIR / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
df = pd.read_csv(DATA_PATH)

df.shape

(16338, 6)

In [4]:
required_columns = [
    "text_raw",
    "text_basic_clean",
    "text_clean",
    "type",
    "queue",
    "priority"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

In [5]:
df = df.copy()

for column in [
    "text_raw",
    "text_basic_clean",
    "text_clean"
]:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
    )

df = df[
    (df["text_basic_clean"].str.strip() != "") &
    (df["text_clean"].str.strip() != "")
].reset_index(drop=True)

df.shape

(16338, 6)

In [6]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df["text_basic_clean"]
)

tfidf_matrix.shape

(16338, 53194)

### Cosine Similarity

Cosine similarity measures how close two text vectors are in direction.

- `1` → very similar
- `0` → little or no similarity

For this project:

Historical ticket → TF-IDF vector
New ticket → TF-IDF vector

The cosine similarity between them becomes the retrieval score.

In [7]:
def find_similar_tickets_tfidf(
    query_text,
    top_k=5,
    exclude_index=None
):
    query_vector = tfidf_vectorizer.transform(
        [query_text]
    )

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).ravel()

    if exclude_index is not None:
        similarity_scores[exclude_index] = -1

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = df.iloc[top_indices].copy()

    results["similarity_score"] = (
        similarity_scores[top_indices]
    )

    return results[
        [
            "text_raw",
            "queue",
            "priority",
            "type",
            "similarity_score"
        ]
    ]

In [8]:
test_index = 100

query_ticket = df.loc[
    test_index,
    "text_basic_clean"
]

tfidf_results = find_similar_tickets_tfidf(
    query_ticket,
    top_k=5,
    exclude_index=test_index
)

tfidf_results

,text_raw,queue,priority,type,similarity_score
259,"Service Interruption Dear Customer Support,\n\...",Technical Support,high,Problem,0.237714
280,Immediate Attention Needed: Cloud-Native SaaS ...,Product Support,medium,Incident,0.229703
293,Performance Concerns on Cloud SaaS Service Dea...,IT Support,low,Incident,0.215196
132,Performance Concerns with Cloud SaaS Platform ...,IT Support,low,Incident,0.200829
217,Performance Concerns with Cloud-Native SaaS Se...,Technical Support,high,Problem,0.197080


In [9]:
query_vector = tfidf_vectorizer.transform(
    [query_ticket]
)

sample_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).ravel()

sample_scores = np.delete(
    sample_scores,
    test_index
)

pd.Series(sample_scores).describe()

count    16337.000000
mean         0.027649
std          0.017954
min          0.000000
25%          0.014916
50%          0.024588
75%          0.036952
max          0.237714
dtype: float64

### Word2Vec Similarity

TF-IDF mainly captures similarity through shared words and phrases.

Word2Vec represents words as dense vectors learned from their surrounding context.

We will represent each ticket by averaging the Word2Vec vectors of the words it contains.

In [10]:
from gensim.models import Word2Vec

In [11]:
sentences = [
    text.split()
    for text in df["text_clean"]
]

len(sentences)

16338

In [12]:
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10,
    seed=42
)

len(word2vec_model.wv)

4038

### Creating a Ticket Representation

Word2Vec gives us a vector for each word.

For example:

`refund → [0.21, -0.15, ...]`

But a ticket contains many words.

To represent the entire ticket with one vector, we average the vectors of all words known to the Word2Vec vocabulary.

This is a simple document-level representation and does not preserve word order.

In [13]:
def document_vector(tokens, model):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if not vectors:
        return np.zeros(
            model.vector_size
        )

    return np.mean(
        vectors,
        axis=0
    )


document_vectors = np.vstack([
    document_vector(
        tokens,
        word2vec_model
    )
    for tokens in sentences
])

document_vectors.shape

(16338, 100)

In [14]:
def find_similar_tickets_word2vec(
    query_text,
    top_k=5,
    exclude_index=None
):
    query_tokens = query_text.split()

    query_vector = document_vector(
        query_tokens,
        word2vec_model
    ).reshape(1, -1)

    similarity_scores = cosine_similarity(
        query_vector,
        document_vectors
    ).ravel()

    if exclude_index is not None:
        similarity_scores[exclude_index] = -1

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = df.iloc[top_indices].copy()

    results["similarity_score"] = (
        similarity_scores[top_indices]
    )

    return results[
        [
            "text_raw",
            "queue",
            "priority",
            "type",
            "similarity_score"
        ]
    ]

In [15]:
query_ticket_clean = df.loc[
    test_index,
    "text_clean"
]

word2vec_results = find_similar_tickets_word2vec(
    query_ticket_clean,
    top_k=5,
    exclude_index=test_index
)

word2vec_results

,text_raw,queue,priority,type,similarity_score
221,Performance Concern with Cloud-Native SaaS Ser...,Technical Support,high,Problem,0.966501
67,"Issue Report Dear Customer Support Team,\n\nI ...",Technical Support,medium,Incident,0.965381
58,"Issue Notification Dear Customer Support Team,...",Technical Support,medium,Incident,0.964639
217,Performance Concerns with Cloud-Native SaaS Se...,Technical Support,high,Problem,0.963324
225,Critical: Agile Team Event Affecting SaaS Serv...,General Inquiry,low,Incident,0.961606


In [16]:
tfidf_display = (
    tfidf_results
    .copy()
)

tfidf_display.insert(
    0,
    "rank",
    range(1, len(tfidf_display) + 1)
)

tfidf_display

word2vec_display = (
    word2vec_results
    .copy()
)

word2vec_display.insert(
    0,
    "rank",
    range(1, len(word2vec_display) + 1)
)

word2vec_display

,rank,text_raw,queue,priority,type,similarity_score
221,1,Performance Concern with Cloud-Native SaaS Ser...,Technical Support,high,Problem,0.966501
67,2,"Issue Report Dear Customer Support Team,\n\nI ...",Technical Support,medium,Incident,0.965381
58,3,"Issue Notification Dear Customer Support Team,...",Technical Support,medium,Incident,0.964639
217,4,Performance Concerns with Cloud-Native SaaS Se...,Technical Support,high,Problem,0.963324
225,5,Critical: Agile Team Event Affecting SaaS Serv...,General Inquiry,low,Incident,0.961606


In [17]:
comparison = pd.DataFrame({
    "TF-IDF Queue": tfidf_results["queue"].values,
    "Word2Vec Queue": word2vec_results["queue"].values
})

comparison

,TF-IDF Queue,Word2Vec Queue
0,Technical Support,Technical Support
1,Product Support,Technical Support
2,IT Support,Technical Support
3,IT Support,Technical Support
4,Technical Support,General Inquiry


### TF-IDF vs Word2Vec

**TF-IDF**

Captures lexical similarity.

If two tickets use similar words or phrases, they are likely to receive a higher similarity score.

Advantages:
- simple
- fast
- interpretable
- strong baseline for text retrieval

**Word2Vec**

Represents words using dense vectors learned from context.

It can capture relationships between words even when exact word overlap is lower.

However, our document representation is created by averaging word vectors, so word order and detailed context are lost.

Therefore, the two methods provide complementary approaches to historical ticket retrieval

In [19]:
tfidf_vectorizer_path = (
    MODELS_DIR / "similarity_tfidf.joblib"
)

joblib.dump(
    tfidf_vectorizer,
    tfidf_vectorizer_path
)

['..\\models\\similarity_tfidf.joblib']

In [20]:
from scipy.sparse import save_npz

tfidf_matrix_path = (
    MODELS_DIR / "similarity_tfidf_matrix.npz"
)

save_npz(
    tfidf_matrix_path,
    tfidf_matrix
)

In [21]:
word2vec_path = (
    MODELS_DIR / "similarity_word2vec.model"
)

word2vec_model.save(
    str(word2vec_path)
)

In [22]:
document_vectors_path = (
    MODELS_DIR / "similarity_document_vectors.npy"
)

np.save(
    document_vectors_path,
    document_vectors
)

In [23]:
metadata = df[
    [
        "text_raw",
        "text_basic_clean",
        "text_clean",
        "type",
        "queue",
        "priority"
    ]
].copy()

metadata_path = (
    MODELS_DIR / "similarity_ticket_metadata.csv"
)

metadata.to_csv(
    metadata_path,
    index=False
)

In [24]:
similarity_files = [
    tfidf_vectorizer_path,
    tfidf_matrix_path,
    word2vec_path,
    document_vectors_path,
    metadata_path
]

artifact_status = pd.DataFrame({
    "file": [
        path.name
        for path in similarity_files
    ],
    "exists": [
        path.exists()
        for path in similarity_files
    ]
})

artifact_status

,file,exists
0,similarity_tfidf.joblib,True
1,similarity_tfidf_matrix.npz,True
2,similarity_word2vec.model,True
3,similarity_document_vectors.npy,True
4,similarity_ticket_metadata.csv,True


In [25]:
new_ticket = """
I was charged twice for the same transaction
and I want one of the payments refunded.
"""

new_ticket_clean = re.sub(
    r"[^a-zA-Z0-9\s]",
    " ",
    new_ticket.lower()
)

new_ticket_clean = re.sub(
    r"\s+",
    " ",
    new_ticket_clean
).strip()

In [26]:
new_ticket_tfidf_results = (
    find_similar_tickets_tfidf(
        new_ticket_clean,
        top_k=5
    )
)

new_ticket_tfidf_results

,text_raw,queue,priority,type,similarity_score
6030,Concern Regarding Monthly Subscription Charges...,Billing and Payments,medium,Problem,0.227160
9764,I was charged twice for the subscription perio...,Billing and Payments,high,Incident,0.227115
12346,Concern About Monthly Subscription Charges Dea...,Billing and Payments,medium,Problem,0.226532
13822,Problem with Monthly Subscription Overcharging...,Billing and Payments,high,Problem,0.208312
15057,Problem with Monthly Subscription Being Charge...,Billing and Payments,high,Problem,0.204812


In [27]:
new_ticket_w2v_results = (
    find_similar_tickets_word2vec(
        new_ticket_clean,
        top_k=5
    )
)

new_ticket_w2v_results

,text_raw,queue,priority,type,similarity_score
15136,Problem with subscription renewal payment My s...,Billing and Payments,high,Problem,0.847928
13036,Problem with Subscription Renewal Payment My s...,Billing and Payments,high,Problem,0.847291
6142,Issue with Monthly Subscription Fee Deduction ...,Billing and Payments,high,Problem,0.843422
6391,Issue with Monthly Subscription Payment The mo...,Billing and Payments,high,Problem,0.838489
235,Query About Recent Invoice Payment Fees Dear C...,Billing and Payments,medium,Request,0.830158


In [28]:
final_comparison = pd.DataFrame({
    "TF-IDF Queue": new_ticket_tfidf_results["queue"].values,
    "TF-IDF Score": new_ticket_tfidf_results["similarity_score"].round(4).values,
    "Word2Vec Queue": new_ticket_w2v_results["queue"].values,
    "Word2Vec Score": new_ticket_w2v_results["similarity_score"].round(4).values
})

final_comparison

,TF-IDF Queue,TF-IDF Score,Word2Vec Queue,Word2Vec Score
0,Billing and Payments,0.2272,Billing and Payments,0.8479
1,Billing and Payments,0.2271,Billing and Payments,0.8473
2,Billing and Payments,0.2265,Billing and Payments,0.8434
3,Billing and Payments,0.2083,Billing and Payments,0.8385
4,Billing and Payments,0.2048,Billing and Payments,0.8302
